In [ ]:
import csv
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import r2_score
from torch.cuda.amp import GradScaler, autocast
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool


# Random seed configuration
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Model definitions
class SimpleGNN(nn.Module):
    """Pretrained GCN teacher for a single stratification."""

    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)

        if edge_dim:
            self.edge_norm = nn.BatchNorm1d(edge_dim)

        if global_dim:
            self.global_norm = nn.BatchNorm1d(global_dim)
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            )

        self.convs = nn.ModuleList()
        in_dim = node_dim
        for hidden_dim in hidden_dims:
            self.convs.append(GCNConv(in_dim, hidden_dim))
            in_dim = hidden_dim

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1),
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)

        if hasattr(self, "edge_norm") and getattr(data, "edge_attr", None) is not None:
            _ = self.edge_norm(data.edge_attr)

        u = getattr(data, "u", None)
        if hasattr(self, "global_norm") and u is not None:
            u = self.global_norm(u)

        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)

        node_pool = global_mean_pool(x, data.batch)
        h = (
            torch.cat([node_pool, self.global_mlp(u)], dim=1)
            if u is not None
            else node_pool
        )

        out = self.output_mlp(h).view(-1)
        return (out, h) if return_feat else out

class EnhancedGNN(nn.Module):
    """GCN student used for single-teacher knowledge distillation."""

    def __init__(self, node_dim, edge_dim, global_dim, hidden_dims, dropout=0.2):
        super().__init__()
        self.node_norm = nn.BatchNorm1d(node_dim)
        self.edge_norm = nn.BatchNorm1d(edge_dim) if edge_dim else None
        self.global_norm = nn.BatchNorm1d(global_dim) if global_dim else None

        if global_dim:
            self.global_mlp = nn.Sequential(
                nn.Linear(global_dim, hidden_dims[-1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            )

        self.convs = nn.ModuleList()
        in_dim = node_dim
        for hidden_dim in hidden_dims:
            self.convs.append(GCNConv(in_dim, hidden_dim))
            in_dim = hidden_dim

        self.dropout = nn.Dropout(dropout)
        self.final_dim = hidden_dims[-1] * (2 if global_dim else 1)
        self.output_mlp = nn.Sequential(
            nn.Linear(self.final_dim, self.final_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(self.final_dim // 2, 1),
        )

    def forward(self, data, return_feat=False):
        x = self.node_norm(data.x)

        if self.edge_norm is not None and getattr(data, "edge_attr", None) is not None:
            _ = self.edge_norm(data.edge_attr)

        u = getattr(data, "u", None)
        if u is not None:
            u = self.global_norm(u)
            global_feat = self.global_mlp(u)

        for conv in self.convs:
            x = F.relu(conv(x, data.edge_index))
            x = self.dropout(x)

        pooled = global_mean_pool(x, data.batch)
        h = torch.cat([pooled, global_feat], dim=1) if u is not None else pooled
        out = self.output_mlp(h).view(-1)
        return (out, h) if return_feat else out

class Adapter(nn.Module):
    """Project student graph features onto the teacher feature dimension."""

    def __init__(self, dim_s, dim_t):
        super().__init__()
        self.linear = nn.Linear(dim_s, dim_t)

    def forward(self, h):
        return self.linear(h)

# Graph data loader
def create_data_loader(graph_list, batch_size=32, shuffle=True):
    """Convert stored graph dictionaries to PyG mini-batches."""
    data_list = []
    for graph in graph_list:
        data_list.append(
            Data(
                x=graph["x"],
                edge_index=graph["edge_index"],
                edge_attr=graph.get("edge_attr", None),
                u=graph.get("u", None),
                y=graph["y"],
            )
        )

    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

# Validation metric for checkpoint selection
def evaluate_validation_r2(model, loader, device):
    """Compute R² on the selection split in normalized target units."""
    model.eval()
    observed = []
    predicted = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            observed.append(batch.y.view(-1).cpu().numpy())
            predicted.append(model(batch).view(-1).cpu().numpy())
    return float(r2_score(np.concatenate(observed), np.concatenate(predicted)))

# Single-teacher distillation trainer
def train_model_single_teacher(
    train_dir,
    val_dir,
    teacher_path,
    save_path,
    hint_lambda=5.0,
    weight_ratio=(0.6, 0.4),
    hidden_dims=None,
    dropout=0.1,
    epochs=500,
    batch_size=64,
    lr=1e-3,
    min_lr=1e-4,
    lr_patience=20,
    es_patience=50,
    seed=42,
):
    """Train a student with the original three-term single-teacher loss.

    The selection split is used for checkpoint selection and early stopping.
    This routine does not report independent test-set performance.
    """
    if hidden_dims is None:
        hidden_dims = [128, 128]

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_seed(seed)

    print(f"Using device: {device}")

    train_graphs = torch.load(os.path.join(train_dir, "graph_data.pt"))
    val_graphs = torch.load(os.path.join(val_dir, "graph_data.pt"))

    train_targets = torch.stack([g["y"] for g in train_graphs]).view(-1)
    y_mean = train_targets.mean().item()
    y_std = train_targets.std().item() + 1e-8

    for graph in train_graphs:
        graph["y"] = (graph["y"] - y_mean) / y_std

    for graph in val_graphs:
        graph["y"] = (graph["y"] - y_mean) / y_std

    train_loader = create_data_loader(train_graphs, batch_size, shuffle=True)
    val_loader = create_data_loader(val_graphs, batch_size, shuffle=False)

    sample = train_graphs[0]
    node_dim = sample["x"].size(1)
    edge_dim = (
        sample.get("edge_attr", None).size(1)
        if sample.get("edge_attr", None) is not None
        else 0
    )
    global_dim = (
        sample.get("u", None).size(1)
        if sample.get("u", None) is not None
        else 0
    )

    student = EnhancedGNN(
        node_dim=node_dim,
        edge_dim=edge_dim,
        global_dim=global_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    teacher_checkpoint = torch.load(teacher_path, map_location=device)
    teacher = SimpleGNN(
        teacher_checkpoint["node_dim"],
        teacher_checkpoint.get("edge_dim", 0),
        teacher_checkpoint.get("global_dim", 0),
        teacher_checkpoint["hidden_dims"],
        teacher_checkpoint["dropout"],
    ).to(device)

    teacher.load_state_dict(
        teacher_checkpoint["model_state_dict"],
        strict=False,
    )
    teacher.eval()

    for parameter in teacher.parameters():
        parameter.requires_grad = False

    adapter = Adapter(student.final_dim, teacher.final_dim).to(device)

    optimizer = optim.Adam(
        list(student.parameters()) + list(adapter.parameters()),
        lr=lr,
        weight_decay=1e-5,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=lr_patience,
        min_lr=min_lr,
    )
    scaler = GradScaler(enabled=(device.type == "cuda"))

    best_val_r2 = -np.inf
    best_epoch = 0
    patience_count = 0

    history = {"epoch": [], "loss": [], "learning_rate": [], "val_r2": []}

    for epoch in range(1, epochs + 1):
        student.train()
        adapter.train()
        total_loss = 0.0

        for batch in train_loader:
            batch = batch.to(device)

            with torch.no_grad():
                teacher_prediction, teacher_feature = teacher(
                    batch,
                    return_feat=True,
                )
                teacher_prediction = teacher_prediction.view(-1)

            with autocast(enabled=(device.type == "cuda")):
                student_prediction, student_feature = student(
                    batch,
                    return_feat=True,
                )
                student_prediction = student_prediction.view(-1)

                fused_prediction = (
                    weight_ratio[0] * student_prediction
                    + weight_ratio[1] * teacher_prediction
                )

                hint_loss = F.mse_loss(
                    adapter(student_feature),
                    teacher_feature,
                )

                loss = (
                    weight_ratio[0]
                    * F.mse_loss(fused_prediction, batch.y.view(-1))
                    + weight_ratio[1]
                    * F.mse_loss(student_prediction, teacher_prediction)
                    + hint_lambda * hint_loss
                )

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        average_loss = total_loss / max(len(train_loader), 1)
        current_lr = optimizer.param_groups[0]["lr"]

        # Validation R² is retained only for checkpoint selection and early stopping.
        val_r2 = evaluate_validation_r2(student, val_loader, device)
        history["epoch"].append(epoch)
        history["loss"].append(average_loss)
        history["learning_rate"].append(current_lr)
        history["val_r2"].append(val_r2)

        if epoch == 1 or epoch % 30 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:4d} | Loss {average_loss:.6f} | "
                f"Val R2 {val_r2:.4f} | LR {current_lr:.2e}"
            )

        scheduler.step(average_loss)

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_epoch = epoch
            patience_count = 0

            torch.save(
                {
                    "model_state_dict": student.state_dict(),
                    "adapter_state_dict": adapter.state_dict(),
                    "y_mean": y_mean,
                    "y_std": y_std,
                    "history": history,
                    "node_dim": node_dim,
                    "edge_dim": edge_dim,
                    "global_dim": global_dim,
                    "hidden_dims": hidden_dims,
                    "dropout": dropout,
                    "teacher_path": teacher_path,
                    "hint_lambda": hint_lambda,
                    "weight_ratio": weight_ratio,
                    "best_epoch": best_epoch,
                    "best_val_r2": best_val_r2,
                },
                save_path,
            )

            if epoch == 1 or epoch % 30 != 0:
                print(
                    f"Saved best model at epoch {epoch}, "
                    f"validation R2 = {best_val_r2:.4f}"
                )
        else:
            patience_count += 1

            if patience_count >= es_patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    print(f"Training complete; best validation R2 = {best_val_r2:.4f}")
    return save_path

# Main experiment: single-teacher distillation
if __name__ == "__main__":
    seeds = [0, 8, 42, 100, 456, 618, 1189, 2025, 2077, 2048]
    teacher_order = ("qcut", "elem", "molwt", "fp", "scaffold")

    # Fixed configurations are indexed by (teacher, seed); no grid search is run here.
    PROJECT_ROOT = os.path.abspath(os.getcwd())
    config_path = os.path.join(PROJECT_ROOT, "config", "single_teacher_seed_hyperparameters.csv")
    seed_teacher_config = {}
    with open(config_path, newline="", encoding="utf-8") as config_file:
        for row in csv.DictReader(config_file):
            teacher_name = row["teacher"].strip()
            seed = int(row["seed"])
            key = (teacher_name, seed)
            if key in seed_teacher_config:
                raise ValueError(f"Duplicate configuration for teacher={teacher_name}, seed={seed}.")
            required = ("hint_lambda", "weight_student", "weight_teacher")
            if any(not row[name].strip() for name in required):
                raise ValueError(
                    f"Missing verified hyperparameters for teacher={teacher_name}, seed={seed}. "
                    "Fill in config/single_teacher_seed_hyperparameters.csv using the original experiment records."
                )
            hint_lambda = float(row["hint_lambda"])
            weight_ratio = (float(row["weight_student"]), float(row["weight_teacher"]))
            if not np.isfinite(hint_lambda) or not all(np.isfinite(w) for w in weight_ratio):
                raise ValueError(f"Non-finite hyperparameters for teacher={teacher_name}, seed={seed}.")
            config_id = row["config_id"].strip() or (
                f"hl{hint_lambda:g}_wr{weight_ratio[0]:g}_{weight_ratio[1]:g}"
            )
            seed_teacher_config[key] = {
                "hint_lambda": hint_lambda,
                "weight_ratio": weight_ratio,
                "config_id": config_id,
            }

    expected = {(teacher_name, seed) for teacher_name in teacher_order for seed in seeds}
    if set(seed_teacher_config) != expected:
        missing = sorted(expected - set(seed_teacher_config))
        extra = sorted(set(seed_teacher_config) - expected)
        raise ValueError(f"Configuration keys do not match teacher/seed combinations; missing={missing}, extra={extra}.")

    train_dir = os.path.join(PROJECT_ROOT, "data-set", "train")
    val_dir = os.path.join(PROJECT_ROOT, "data-set", "validation")
    teacher_checkpoint_dir = os.path.join(PROJECT_ROOT, "checkpoints", "teachers")
    save_root = os.path.join(PROJECT_ROOT, "results", "single_teacher_distillation")
    os.makedirs(save_root, exist_ok=True)

    epochs = 1000
    batch_size = 64
    lr = 1e-3
    min_lr = 5e-5
    lr_patience = 30
    es_patience = 100
    hidden_dims = [128, 128]
    dropout = 0.1

    # Record checkpoints without a separate prediction-metrics report.
    manifest_path = os.path.join(save_root, "checkpoint_manifest.csv")
    with open(manifest_path, "w", newline="", encoding="utf-8") as manifest_file:
        writer = csv.DictWriter(
            manifest_file,
            fieldnames=["teacher", "seed", "config_id", "checkpoint_path"],
        )
        writer.writeheader()
        for teacher_name in teacher_order:
            teacher_path = os.path.join(teacher_checkpoint_dir, f"{teacher_name}.pt")
            teacher_output_dir = os.path.join(save_root, teacher_name)
            os.makedirs(teacher_output_dir, exist_ok=True)
            for seed in seeds:
                cfg = seed_teacher_config[(teacher_name, seed)]
                save_path = os.path.join(
                    teacher_output_dir,
                    f"student_{cfg['config_id']}_seed{seed}.pt",
                )
                print(f"Teacher={teacher_name} | Seed={seed} | Configuration={cfg['config_id']}")
                train_model_single_teacher(
                    train_dir=train_dir,
                    val_dir=val_dir,
                    teacher_path=teacher_path,
                    save_path=save_path,
                    hint_lambda=cfg["hint_lambda"],
                    weight_ratio=cfg["weight_ratio"],
                    hidden_dims=hidden_dims,
                    dropout=dropout,
                    epochs=epochs,
                    batch_size=batch_size,
                    lr=lr,
                    min_lr=min_lr,
                    lr_patience=lr_patience,
                    es_patience=es_patience,
                    seed=seed,
                )
                writer.writerow({
                    "teacher": teacher_name,
                    "seed": seed,
                    "config_id": cfg["config_id"],
                    "checkpoint_path": os.path.relpath(save_path, PROJECT_ROOT),
                })
                manifest_file.flush()
    print(f"Completed single-teacher distillation. Manifest: {manifest_path}")
